# Evaluation of IDF of hourly CPM emulators

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.furflex_default_params import *

In [ ]:
import dask
import dask.array
from dask.distributed import Client
import IPython
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from mlde_analysis.idf import plot_pmf, calc_pmf_ndimage

In [ ]:
client = Client()
client

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.furflex_magics 
EVAL_DS, MODELS, TARGET_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS["CPM"]

## IDF

In [ ]:
%%time

var = "pr"
pmfs = {}

pmfs["CPM"] = TARGET_DAS[var].groupby(["time.season", "time.year"]).map(calc_pmf_ndimage).mean(dim=["season", "year"])

for model, model_da in PRED_DAS[var].groupby(["model"]):
    pmfs[model] = model_da.squeeze().groupby(["time.season", "time.year"]).map(calc_pmf_ndimage).mean(dim=["season", "year"])

In [ ]:
%%time

entries = list(pmfs.keys())
cols = 3
npads = (cols - len(entries) % cols) % cols
grid_spec = np.pad(np.array(entries), (0,npads), mode='constant', constant_values=".").reshape(-1, cols)

fig = plt.figure(layout="constrained", figsize=(2*grid_spec.shape[1]+1, 2*grid_spec.shape[0]))
axd = fig.subplot_mosaic(grid_spec, sharex=True, sharey=True)

for model, pmf in pmfs.items():
    ax = axd[model]
    shw = plot_pmf(ax, pmf, title=model, norm=matplotlib.colors.LogNorm(vmin=1e-5, vmax=0.1))

cb = fig.colorbar(
    shw,
    ax=axd.values(),
    location="right",
    extend="max",
)
cb.set_label("Probability mass", fontsize="small")
cb.ax.tick_params(labelsize="small")

# for label in grid_spec[:,0]:
#     if label == ".":
#         continue
#     axd[label].set_ylabel("Spell duration (hours)")

# for label in grid_spec[-1,:]:
#     if label == ".":
#         continue
#     axd[label].set_xlabel("Maximum intensity (mm/hr)")

fig.supxlabel("Maximum intensity (mm/hr)")
fig.supylabel("Spell duration (hours)")

plt.show()

In [ ]:
%%time

entries = [ k for k in pmfs.keys() ]# if k != "CPM" ]
cols = 3
npads = (cols - len(entries) % cols) % cols
grid_spec = np.pad(np.array(entries), (0,npads), mode='constant', constant_values=".").reshape(-1, cols)

fig = plt.figure(layout="constrained", figsize=(2*grid_spec.shape[1]+1, 2*grid_spec.shape[0]+0.5))
axd = fig.subplot_mosaic(grid_spec, sharex=True, sharey=True)

cpm_pmf = pmfs["CPM"]

for model, pmf in pmfs.items():
    if model == "CPM":
        shw = plot_pmf(axd[model], pmf, title=model, norm=matplotlib.colors.LogNorm(vmin=1e-5, vmax=0.1))
        cb = fig.colorbar(
            shw,
            ax=axd[model],
            location="right",
            extend="max",
        )
        cb.set_label("Probability mass", fontsize="small")
        cb.ax.tick_params(labelsize="x-small")
    else:
        ax = axd[model]
        shw = plot_pmf(ax, pmf - cpm_pmf, title=model, cmap="RdBu", norm=matplotlib.colors.SymLogNorm(linthresh=0.0001, vmin=-0.1, vmax=0.1))

cb = fig.colorbar(
    shw,
    ax=[ax for k,ax in axd.items() if k != "CPM"],
    location="right",
    extend="both",
)
cb.set_label("Probability mass error", fontsize="small")
cb.ax.tick_params(labelsize="small")

fig.supxlabel("Maximum intensity (mm/hr)")
fig.supylabel("Spell duration (hours)")

plt.show()

In [ ]:
%%time

entries = [ k for k in pmfs.keys() ]# if k != "CPM" ]
cols = 3
npads = (cols - len(entries) % cols) % cols
grid_spec = np.pad(np.array(entries), (0,npads), mode='constant', constant_values=".").reshape(-1, cols)

fig = plt.figure(layout="constrained", figsize=(2*grid_spec.shape[1]+1, 2*grid_spec.shape[0]+0.5))
axd = fig.subplot_mosaic(grid_spec, sharex=True, sharey=True)

cpm_pmf = pmfs["CPM"]

norm = matplotlib.colors.LogNorm(vmin=1e-5, vmax=0.1)
diff_norm = matplotlib.colors.SymLogNorm(linthresh=0.0001, vmin=-500, vmax=500)

for model, pmf in pmfs.items():
    if model == "CPM":
        shw = plot_pmf(axd[model], pmf, title=model, norm=norm)
        cb = fig.colorbar(
            shw,
            ax=axd[model],
            location="right",
            extend="max",
        )
        cb.set_label("Probability mass", fontsize="small")
        cb.ax.tick_params(labelsize="x-small")
    else:
        ax = axd[model]
        shw = plot_pmf(ax, 100*(pmf - cpm_pmf)/cpm_pmf, title=model, cmap="RdBu")#, norm=diff_norm)

cb = fig.colorbar(
    shw,
    ax=[ax for k,ax in axd.items() if k != "CPM"],
    location="right",
    # extend="both",
)
cb.set_label("Probability mass relative error", fontsize="small")
cb.ax.tick_params(labelsize="small")

fig.supxlabel("Maximum intensity (mm/hr)")
fig.supylabel("Spell duration (hours)")

plt.show()

In [ ]:
client.close()